# Converting a ROOT dataset to NPZ and HDF5

This notebook walks through the conversion of a ROOT dataset into two formats used by the training scripts: NPZ and HDF5. We start by reading the columns with `ROOT.RDataFrame`, assemble the feature matrix `X` and label vector `y`, and write both files. The last section collects the operations into reusable functions.

The input is a ROOT `RNTuple` with one target column and numeric feature columns. The output schema is `X` (2-D `float32`) and `y` (1-D `int64`). Columns with unsupported ROOT types are left out. Both files store the same converted arrays.

## Source file and conversion settings

The source below is the normalized RNTuple, stored under the object name `tree`. The notebook uses a separate output directory so that inspecting the conversion does not overwrite any existing datasets. Run the cells in order from the project directory containing `data/original/`.

In [1]:
import os
import json
import zipfile
from pathlib import Path

import numpy as np
import h5py
import ROOT

root_path = "data/original/mc_normalized_rntuple.root"
tree_name = "tree"
target_col = "Label"

output_dir = Path("data/conversion_walkthrough")
output_npz = output_dir / "mc_normalized_rntuple.npz"
output_h5 = output_dir / "mc_normalized_rntuple.h5"

NUMERIC_TYPES = {
    "float", "double", "Float_t", "Double_t", "int",
    "unsigned int", "long", "unsigned long", "long long",
    "unsigned long long", "std::int32_t", "std::uint32_t",
    "std::int64_t", "std::uint64_t",
}
X_DTYPE = np.float32
Y_DTYPE = np.int64
AUTO_CHUNK_TARGET_BYTES = 1024 * 1024

if not Path(root_path).is_file():
    raise FileNotFoundError(f"Cannot find {root_path}. Current directory: {Path.cwd()}")

print("ROOT version:", ROOT.gROOT.GetVersion())
print("Source:", root_path)
print("NPZ output:", output_npz)
print("HDF5 output:", output_h5)

ROOT version: 6.40.00
Source: data/original/mc_normalized_rntuple.root
NPZ output: data/conversion_walkthrough/mc_normalized_rntuple.npz
HDF5 output: data/conversion_walkthrough/mc_normalized_rntuple.h5


/home/satt/.local/lib/python3.14/site-packages/ROOT/__init__.py:222: UserWarning: 
This distribution of ROOT is in alpha stage. Feedback is welcome and appreciated. Feel free to reach out to the user forum for questions and general feedback at https://root-forum.cern.ch or to submit an issue at https://github.com/root-project/root/issues. Do not rely on this distribution for production purposes.

  warnings.warn(


## Read the ROOT schema

`RDataFrame` exposes the columns of the `tree` object. Creating it does not materialize every event in NumPy. We first get the available column names and check that the target exists.

In [2]:
rdf = ROOT.RDataFrame(tree_name, root_path)
columns = [str(c) for c in rdf.GetColumnNames()]

if target_col not in columns:
    raise ValueError(f"Missing target {target_col!r}. Found: {columns}")

print("ROOT columns:", len(columns))
for name in columns:
    print(" ", name)

ROOT columns: 20
  Label
  lep1_E_norm
  lep1_eta_norm
  lep1_phi_norm
  lep1_pt_norm
  lep2_E_norm
  lep2_eta_norm
  lep2_phi_norm
  lep2_pt_norm
  lep3_E_norm
  lep3_eta_norm
  lep3_phi_norm
  lep3_pt_norm
  lep4_E_norm
  lep4_eta_norm
  lep4_phi_norm
  lep4_pt_norm
  m4l_norm
  sample_category
  weight


The type reported by `GetColumnType()` determines whether a column can enter the numeric matrix. It also lets us compare the ROOT schema with the types saved later.

In [3]:
root_column_types = {}
for name in columns:
    try:
        root_column_types[name] = str(rdf.GetColumnType(name))
    except Exception:
        root_column_types[name] = "unknown"

for name in columns:
    print(f"{name:24s} {root_column_types[name]}")

Label                    std::int32_t
lep1_E_norm              double
lep1_eta_norm            double
lep1_phi_norm            double
lep1_pt_norm             double
lep2_E_norm              double
lep2_eta_norm            double
lep2_phi_norm            double
lep2_pt_norm             double
lep3_E_norm              double
lep3_eta_norm            double
lep3_phi_norm            double
lep3_pt_norm             double
lep4_E_norm              double
lep4_eta_norm            double
lep4_phi_norm            double
lep4_pt_norm             double
m4l_norm                 double
sample_category          std::string
weight                   double


### Select the columns

Keep the target and the columns with a type listed in `NUMERIC_TYPES`. A string column such as `sample_category` is skipped. `weight` was maintained as a feature in this workflow.

In [4]:
selected_cols = []
skipped_cols = []

for name in columns:
    ctype = root_column_types[name]
    if name == target_col or ctype in NUMERIC_TYPES:
        selected_cols.append(name)
    else:
        skipped_cols.append((name, ctype))

feature_names = [name for name in selected_cols if name != target_col]

print("Target:", target_col)
print("Selected features:", len(feature_names))
for name in feature_names:
    print(" ", name)
print("Skipped:", skipped_cols)

Target: Label
Selected features: 18
  lep1_E_norm
  lep1_eta_norm
  lep1_phi_norm
  lep1_pt_norm
  lep2_E_norm
  lep2_eta_norm
  lep2_phi_norm
  lep2_pt_norm
  lep3_E_norm
  lep3_eta_norm
  lep3_phi_norm
  lep3_pt_norm
  lep4_E_norm
  lep4_eta_norm
  lep4_phi_norm
  lep4_pt_norm
  m4l_norm
  weight
Skipped: [('sample_category', 'std::string')]


## Materialize the selected columns

`AsNumpy()` triggers the ROOT read for the requested columns. It returns a dictionary with one NumPy array per column. The call below reads all selected rows into memory in a single operation; the code does not iterate over chunks or training batches.

In [5]:
data = rdf.AsNumpy([target_col] + feature_names)

for name in [target_col] + feature_names:
    values = np.asarray(data[name])
    print(f"{name:24s} shape={str(values.shape):15s} dtype={str(values.dtype):8s} first={values[:2].tolist()}")

Label                    shape=(757448,)       dtype=int32    first=[1, 1]
lep1_E_norm              shape=(757448,)       dtype=float64  first=[-0.5206095358512821, -0.5098569045944664]
lep1_eta_norm            shape=(757448,)       dtype=float64  first=[-0.8182047845254599, -1.0894743943414058]
lep1_phi_norm            shape=(757448,)       dtype=float64  first=[-1.532390382313856, 0.247622408395777]
lep1_pt_norm             shape=(757448,)       dtype=float64  first=[-0.525539489457196, -0.7314453669975617]
lep2_E_norm              shape=(757448,)       dtype=float64  first=[-0.4443357543651154, -0.5893454690046364]
lep2_eta_norm            shape=(757448,)       dtype=float64  first=[-0.7167323248548291, -0.34585082124326605]
lep2_phi_norm            shape=(757448,)       dtype=float64  first=[0.2323329148562538, -1.2121856935888509]
lep2_pt_norm             shape=(757448,)       dtype=float64  first=[-0.30207541097228224, -0.33115302063515045]
lep3_E_norm              shape=(757448,

For this dataset the numeric ROOT features are `double`, which normally appears as NumPy `float64`. `Label` is `std::int32_t` and normally appears as `int32`. The next operations convert those column arrays into the types expected by the training data loaders.

## Build the target vector

The target is extracted independently of the feature columns. `np.asarray()` retrieves the materialized column as a NumPy array; `.astype(np.int64)` creates the label vector with a uniform integer type.

In [6]:
y_before_cast = np.asarray(data[target_col])
y = y_before_cast.astype(Y_DTYPE)

print("Target before:", y_before_cast.shape, y_before_cast.dtype)
print("Target after: ", y.shape, y.dtype)
print("First labels:", y[:5].tolist())
print("Values unchanged:", np.array_equal(y_before_cast, y))

Target before: (757448,) int32
Target after:  (757448,) int64
First labels: [1, 1, 1, 1, 1]
Values unchanged: True


## Assemble the feature matrix

`AsNumpy()` keeps each feature in a separate one-dimensional array. We collect those arrays in `feature_names` order before stacking them into a matrix. Each event becomes one row, and each feature becomes one column.

In [7]:
feature_arrays = [np.asarray(data[name]) for name in feature_names]

for name, values in zip(feature_names, feature_arrays):
    print(f"{name:24s} {str(values.shape):15s} {values.dtype}")

lep1_E_norm              (757448,)       float64
lep1_eta_norm            (757448,)       float64
lep1_phi_norm            (757448,)       float64
lep1_pt_norm             (757448,)       float64
lep2_E_norm              (757448,)       float64
lep2_eta_norm            (757448,)       float64
lep2_phi_norm            (757448,)       float64
lep2_pt_norm             (757448,)       float64
lep3_E_norm              (757448,)       float64
lep3_eta_norm            (757448,)       float64
lep3_phi_norm            (757448,)       float64
lep3_pt_norm             (757448,)       float64
lep4_E_norm              (757448,)       float64
lep4_eta_norm            (757448,)       float64
lep4_phi_norm            (757448,)       float64
lep4_pt_norm             (757448,)       float64
m4l_norm                 (757448,)       float64
weight                   (757448,)       float64


In [8]:
X_before_cast = np.column_stack(feature_arrays)

print("Stacked shape:", X_before_cast.shape)
print("Stacked dtype:", X_before_cast.dtype)
print("First row:", X_before_cast[0].tolist() if len(X_before_cast) else [])

Stacked shape: (757448, 18)
Stacked dtype: float64
First row: [-0.5206095358512821, -0.8182047845254599, -1.532390382313856, -0.525539489457196, -0.4443357543651154, -0.7167323248548291, 0.2323329148562538, -0.30207541097228224, -0.7701669497003744, -0.4218637223604329, 0.1338428271181272, -0.8130696582100554, -0.8614932005357446, 0.22801050327989555, -1.3775477981384445, -0.9978399535101065, -0.6133801525348296, 1.7845057894900682e-05]


### Convert the matrix to `float32`

The feature matrix is stored as `float32`, so each element occupies four bytes. This conversion can round values that were stored as ROOT `double`. `np.ascontiguousarray()` then ensures a C-contiguous layout, with the feature values of each event adjacent in memory.

In [9]:
X_after_cast = X_before_cast.astype(X_DTYPE)
X = np.ascontiguousarray(X_after_cast)

print("Before:", X_before_cast.shape, X_before_cast.dtype)
print("After: ", X.shape, X.dtype)
print("C-contiguous:", X.flags.c_contiguous)

if X.size:
    print("First feature before:", X_before_cast[0, 0])
    print("First feature after: ", X[0, 0])
    print("Absolute difference: ", abs(float(X_before_cast[0, 0]) - float(X[0, 0])))

Before: (757448, 18) float64
After:  (757448, 18) float32
C-contiguous: True
First feature before: -0.5206095358512821
First feature after:  -0.52060956
Absolute difference:  2.177734947927945e-08


At this point the data is in the form that both file writers will use. There is no normalization, event shuffling, batch construction, or train/test split in this procedure. The event order is unchanged, and the selected feature order is recorded separately in JSON metadata.

In [10]:
num_samples, num_features = X.shape

assert y.shape == (num_samples,)
assert X.dtype == np.dtype(X_DTYPE)
assert y.dtype == np.dtype(Y_DTYPE)

print("Events:", num_samples)
print("Features:", num_features)
print(f"X payload: {X.nbytes:,} bytes")
print(f"y payload: {y.nbytes:,} bytes")
print(f"Total numeric payload: {X.nbytes + y.nbytes:,} bytes")

Events: 757448
Features: 18
X payload: 54,536,256 bytes
y payload: 6,059,584 bytes
Total numeric payload: 60,595,840 bytes


## Write the NPZ archive

`np.savez()` places `X.npy` and `y.npy` inside an uncompressed ZIP archive. The feature names are absent from the two arrays; a companion metadata file records their order.

In [11]:
output_dir.mkdir(parents=True, exist_ok=True)
np.savez(output_npz, X=X, y=y)

print("Written:", output_npz)
print(f"Size on disk: {output_npz.stat().st_size:,} bytes")

Written: data/conversion_walkthrough/mc_normalized_rntuple.npz
Size on disk: 60,596,330 bytes


### Record the dataset metadata

The JSON sidecar is stored next to the NPZ file. A reader can inspect the column names and data types without extracting `X.npy` or `y.npy`.

In [12]:
def write_metadata(metadata_path, **fields):
    parent = os.path.dirname(metadata_path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    with open(metadata_path, "w") as f:
        json.dump(fields, f, indent=2)
    return metadata_path

npz_metadata_path = write_metadata(
    str(output_npz) + ".metadata.json",
    dataset=os.path.splitext(os.path.basename(output_npz))[0],
    format="NPZ",
    num_samples=int(num_samples),
    num_features=int(num_features),
    num_classes=int(len(np.unique(y))),
    target_column=target_col,
    feature_columns=feature_names,
    X_dtype=str(np.dtype(X_DTYPE)),
    y_dtype=str(np.dtype(Y_DTYPE)),
)

print(Path(npz_metadata_path).read_text())

{
  "dataset": "mc_normalized_rntuple",
  "format": "NPZ",
  "num_samples": 757448,
  "num_features": 18,
  "num_classes": 2,
  "target_column": "Label",
  "feature_columns": [
    "lep1_E_norm",
    "lep1_eta_norm",
    "lep1_phi_norm",
    "lep1_pt_norm",
    "lep2_E_norm",
    "lep2_eta_norm",
    "lep2_phi_norm",
    "lep2_pt_norm",
    "lep3_E_norm",
    "lep3_eta_norm",
    "lep3_phi_norm",
    "lep3_pt_norm",
    "lep4_E_norm",
    "lep4_eta_norm",
    "lep4_phi_norm",
    "lep4_pt_norm",
    "m4l_norm",
    "weight"
  ],
  "X_dtype": "float32",
  "y_dtype": "int64"
}


The ZIP member information is useful when checking whether compression was applied. `np.savez()` writes members with the `ZIP_STORED` method.

In [13]:
with zipfile.ZipFile(output_npz, "r") as archive:
    for member in archive.infolist():
        method = "ZIP_STORED" if member.compress_type == zipfile.ZIP_STORED else member.compress_type
        print(f"{member.filename}: {member.file_size:,} bytes, method={method}")

with np.load(output_npz, allow_pickle=False) as archive:
    print("Arrays:", archive.files)
    for name in archive.files:
        print(name, archive[name].shape, archive[name].dtype)

X.npy: 54,536,384 bytes, method=ZIP_STORED
y.npy: 6,059,712 bytes, method=ZIP_STORED
Arrays: ['X', 'y']
X (757448, 18) float32
y (757448,) int64


## Write the HDF5 file

HDF5 stores the same `X` and `y` arrays as datasets in one file. We use row-oriented chunks with a target of about 1 MiB per chunk of `X`.

The chunk-row count is calculated from the size of one feature row, `num_features × itemsize`. The same row count is then used for `y`. Chunking controls the storage layout; the `compression=None` setting leaves the chunks uncompressed.

In [14]:
def _auto_chunk_rows(num_features, itemsize, target_bytes=AUTO_CHUNK_TARGET_BYTES):
    bytes_per_row = max(1, num_features * itemsize)
    rows = max(1, target_bytes // bytes_per_row)
    return int(rows)

chunk_rows = "auto"
compression = None
itemsize = np.dtype(X_DTYPE).itemsize

resolved_rows = None
if chunk_rows == "auto":
    resolved_rows = _auto_chunk_rows(num_features, itemsize)
elif chunk_rows is not None:
    resolved_rows = int(chunk_rows)

if resolved_rows is not None:
    resolved_rows = min(resolved_rows, max(1, num_samples))

x_chunks = (resolved_rows, num_features) if resolved_rows else None
y_chunks = (resolved_rows,) if resolved_rows else None

print("Target bytes per X chunk:", AUTO_CHUNK_TARGET_BYTES)
print("Rows per chunk:", resolved_rows)
print("X chunk shape:", x_chunks)
print("y chunk shape:", y_chunks)
print("Compression:", compression)

Target bytes per X chunk: 1048576
Rows per chunk: 14563
X chunk shape: (14563, 18)
y chunk shape: (14563,)
Compression: None


In [15]:
with h5py.File(output_h5, "w") as f:
    f.create_dataset("X", data=X, chunks=x_chunks, compression=compression)
    f.create_dataset("y", data=y, chunks=y_chunks, compression=compression)

print("Written:", output_h5)
print(f"Size on disk: {output_h5.stat().st_size:,} bytes")

Written: data/conversion_walkthrough/mc_normalized_rntuple.h5
Size on disk: 61,753,504 bytes


The HDF5 metadata follows the same schema as the NPZ sidecar and adds the chunk-row count and compression setting.

In [16]:
h5_metadata_path = write_metadata(
    str(output_h5) + ".metadata.json",
    dataset=os.path.splitext(os.path.basename(output_h5))[0],
    format="HDF5",
    num_samples=int(num_samples),
    num_features=int(num_features),
    num_classes=int(len(np.unique(y))),
    target_column=target_col,
    feature_columns=feature_names,
    X_dtype=str(np.dtype(X_DTYPE)),
    y_dtype=str(np.dtype(Y_DTYPE)),
    chunk_rows=resolved_rows,
    compression=compression,
)

print(Path(h5_metadata_path).read_text())

{
  "dataset": "mc_normalized_rntuple",
  "format": "HDF5",
  "num_samples": 757448,
  "num_features": 18,
  "num_classes": 2,
  "target_column": "Label",
  "feature_columns": [
    "lep1_E_norm",
    "lep1_eta_norm",
    "lep1_phi_norm",
    "lep1_pt_norm",
    "lep2_E_norm",
    "lep2_eta_norm",
    "lep2_phi_norm",
    "lep2_pt_norm",
    "lep3_E_norm",
    "lep3_eta_norm",
    "lep3_phi_norm",
    "lep3_pt_norm",
    "lep4_E_norm",
    "lep4_eta_norm",
    "lep4_phi_norm",
    "lep4_pt_norm",
    "m4l_norm",
    "weight"
  ],
  "X_dtype": "float32",
  "y_dtype": "int64",
  "chunk_rows": 14563,
  "compression": null
}


### Inspect the HDF5 storage layout

`nbytes` gives the logical size of a dataset. `get_storage_size()` reports the space allocated for its data in HDF5. With fixed-size, uncompressed chunks, the last partially filled chunk can account for a difference between these figures.

In [17]:
with h5py.File(output_h5, "r") as f:
    for name in ("X", "y"):
        ds = f[name]
        print(f"{name}:")
        print("  shape:", ds.shape)
        print("  dtype:", ds.dtype)
        print("  chunks:", ds.chunks)
        print("  compression:", ds.compression)
        print("  logical bytes:", ds.nbytes)
        print("  stored data bytes:", ds.id.get_storage_size())

X:
  shape: (757448, 18)
  dtype: float32
  chunks: (14563, 18)
  compression: None
  logical bytes: 54536256
  stored data bytes: 55572408
y:
  shape: (757448,)
  dtype: int64
  chunks: (14563,)
  compression: None
  logical bytes: 6059584
  stored data bytes: 6174712


## Check the written data

Read the two outputs and compare them with the in-memory arrays. These comparisons check the conversion results after the `float32` and `int64` casts. They do not imply bit-for-bit equality with the ROOT `double` values.

In [18]:
with np.load(output_npz, allow_pickle=False) as npz, h5py.File(output_h5, "r") as h5:
    x_npz = npz["X"]
    y_npz = npz["y"]
    x_h5 = h5["X"][:]
    y_h5 = h5["y"][:]

    checks = {
        "NPZ X equals memory": np.array_equal(x_npz, X),
        "NPZ y equals memory": np.array_equal(y_npz, y),
        "HDF5 X equals memory": np.array_equal(x_h5, X),
        "HDF5 y equals memory": np.array_equal(y_h5, y),
        "NPZ X equals HDF5 X": np.array_equal(x_npz, x_h5),
        "NPZ y equals HDF5 y": np.array_equal(y_npz, y_h5),
    }

for name, passed in checks.items():
    print(f"{name}: {passed}")
assert all(checks.values())

NPZ X equals memory: True
NPZ y equals memory: True
HDF5 X equals memory: True
HDF5 y equals memory: True
NPZ X equals HDF5 X: True
NPZ y equals HDF5 y: True


To read the features as named columns, combine `X` with the `feature_columns` list saved in the metadata. The first two events are printed below.

In [19]:
for row_idx in range(min(2, num_samples)):
    print(f"\nRow {row_idx}:")
    print(f"  {target_col} = {y[row_idx].item()}")
    for col_idx, name in enumerate(feature_names):
        print(f"  {name} = {X[row_idx, col_idx].item()}")


Row 0:
  Label = 1
  lep1_E_norm = -0.5206095576286316
  lep1_eta_norm = -0.8182047605514526
  lep1_phi_norm = -1.5323903560638428
  lep1_pt_norm = -0.5255395174026489
  lep2_E_norm = -0.4443357586860657
  lep2_eta_norm = -0.7167323231697083
  lep2_phi_norm = 0.23233291506767273
  lep2_pt_norm = -0.30207541584968567
  lep3_E_norm = -0.7701669335365295
  lep3_eta_norm = -0.42186373472213745
  lep3_phi_norm = 0.1338428258895874
  lep3_pt_norm = -0.8130696415901184
  lep4_E_norm = -0.8614932298660278
  lep4_eta_norm = 0.22801050543785095
  lep4_phi_norm = -1.3775477409362793
  lep4_pt_norm = -0.9978399276733398
  m4l_norm = -0.6133801341056824
  weight = 1.7845057300291955e-05

Row 1:
  Label = 1
  lep1_E_norm = -0.5098568797111511
  lep1_eta_norm = -1.0894744396209717
  lep1_phi_norm = 0.24762241542339325
  lep1_pt_norm = -0.7314453721046448
  lep2_E_norm = -0.5893454551696777
  lep2_eta_norm = -0.3458508253097534
  lep2_phi_norm = -1.2121857404708862
  lep2_pt_norm = -0.331153035163879

### File sizes

Compare the actual byte counts rather than rounded `du -h` output. The sizes reflect different numerical widths and storage layouts: ROOT stores the selected features as `double`, whereas the two output files store `float32`. The ROOT file may also contain additional columns and compression.

In [20]:
for path in (Path(root_path), output_npz, output_h5):
    size = path.stat().st_size
    print(f"{path}: {size:,} bytes ({size / 1024**2:.2f} MiB)")

data/original/mc_normalized_rntuple.root: 94,895,126 bytes (90.50 MiB)
data/conversion_walkthrough/mc_normalized_rntuple.npz: 60,596,330 bytes (57.79 MiB)
data/conversion_walkthrough/mc_normalized_rntuple.h5: 61,753,504 bytes (58.89 MiB)


## Package the procedure into functions

The working cells above expose each intermediate operation. The same sequence can be placed in functions to convert additional ROOT files from a script. The reader handles column selection and construction of `X` and `y`; each writer receives those arrays and creates its own metadata file.

In [ ]:
def load_xy_from_root(root_path, tree_name, target_col):
    if not os.path.exists(root_path):
        raise FileNotFoundError(f"ROOT file not found: {root_path}")

    rdf = ROOT.RDataFrame(tree_name, root_path)
    columns = [str(c) for c in rdf.GetColumnNames()]
    if target_col not in columns:
        raise ValueError(f"Target {target_col!r} not found. Available: {columns}")

    selected_cols = []
    skipped_cols = []
    for name in columns:
        try:
            ctype = str(rdf.GetColumnType(name))
        except Exception:
            ctype = "unknown"
        if name == target_col or ctype in NUMERIC_TYPES:
            selected_cols.append(name)
        else:
            skipped_cols.append((name, ctype))

    feature_names = [name for name in selected_cols if name != target_col]
    print("Skipped columns:", skipped_cols)
    data = rdf.AsNumpy([target_col] + feature_names)

    y = np.asarray(data[target_col]).astype(Y_DTYPE)
    X = np.ascontiguousarray(
        np.column_stack([np.asarray(data[name]) for name in feature_names]).astype(X_DTYPE)
    )
    return X, y, feature_names

In [ ]:
def write_npz(X, y, feature_names, target_col, output_npz):
    output_npz = os.fspath(output_npz)
    os.makedirs(os.path.dirname(output_npz) or ".", exist_ok=True)
    np.savez(output_npz, X=X, y=y)

    write_metadata(
        output_npz + ".metadata.json",
        dataset=os.path.splitext(os.path.basename(output_npz))[0],
        format="NPZ",
        num_samples=int(len(y)),
        num_features=int(X.shape[1]),
        num_classes=int(len(np.unique(y))),
        target_column=target_col,
        feature_columns=feature_names,
        X_dtype=str(np.dtype(X_DTYPE)),
        y_dtype=str(np.dtype(Y_DTYPE)),
    )
    return output_npz

In [ ]:
def write_hdf5(X, y, feature_names, target_col, output_h5,
               chunk_rows="auto", compression=None):
    output_h5 = os.fspath(output_h5)
    os.makedirs(os.path.dirname(output_h5) or ".", exist_ok=True)
    num_samples, num_features = X.shape

    resolved_rows = None
    if chunk_rows == "auto":
        resolved_rows = _auto_chunk_rows(num_features, np.dtype(X_DTYPE).itemsize)
    elif chunk_rows is not None:
        resolved_rows = int(chunk_rows)
    if resolved_rows is not None:
        resolved_rows = min(resolved_rows, max(1, num_samples))

    x_chunks = (resolved_rows, num_features) if resolved_rows else None
    y_chunks = (resolved_rows,) if resolved_rows else None

    with h5py.File(output_h5, "w") as f:
        f.create_dataset("X", data=X, chunks=x_chunks, compression=compression)
        f.create_dataset("y", data=y, chunks=y_chunks, compression=compression)

    write_metadata(
        output_h5 + ".metadata.json",
        dataset=os.path.splitext(os.path.basename(output_h5))[0],
        format="HDF5",
        num_samples=int(num_samples),
        num_features=int(num_features),
        num_classes=int(len(np.unique(y))),
        target_column=target_col,
        feature_columns=feature_names,
        X_dtype=str(np.dtype(X_DTYPE)),
        y_dtype=str(np.dtype(Y_DTYPE)),
        chunk_rows=resolved_rows,
        compression=compression,
    )
    return output_h5

In [ ]:
def root_to_npz(root_path, tree_name, target_col, output_npz):
    X, y, feature_names = load_xy_from_root(root_path, tree_name, target_col)
    return write_npz(X, y, feature_names, target_col, output_npz)


def root_to_hdf5(root_path, tree_name, target_col, output_h5,
                 chunk_rows="auto", compression=None):
    X, y, feature_names = load_xy_from_root(root_path, tree_name, target_col)
    return write_hdf5(
        X, y, feature_names, target_col, output_h5,
        chunk_rows=chunk_rows, compression=compression,
    )

print("Conversion functions are ready: root_to_npz(), root_to_hdf5()")

The two wrappers read the ROOT source separately when called separately. During the walkthrough, we read it once and passed the same `X` and `y` objects to both writers. That is convenient for checking output equivalence, but it should be considered when measuring conversion time or source I/O.

```python
# Example calls for another dataset:
# root_to_npz("data/1M/mc_normalized_rntuple_1M.root", "tree", "Label", "data/1M/mc_normalized_rntuple_1M.npz")
# root_to_hdf5("data/1M/mc_normalized_rntuple_1M.root", "tree", "Label", "data/1M/mc_normalized_rntuple_1M.h5")
```

The resulting files contain the selected events in the same order, with `X` stored as `float32` and `y` as `int64`. `sample_category` was ommited in this design, while `weight` is retained as a numeric feature. NPZ is written without compression; HDF5 uses the row chunks calculated above with no compression by default.